# Day 08 · 하네스 — 짓고, 그 하네스로 짓는다

7강까지는 손으로 에이전트를 만들었다. 오늘은 **실제 코드 에이전트를 켜고 그 작업 환경을 세팅**한다.
그리고 **하네스를 직접 만들어, 그 하네스로 앱을 짓는다.**

> 같은 모델도 **하네스가 결과를 가른다.**

| 랩 | 무엇을 | 결과물 |
|---|---|---|
| Lab 1 | buggy-request 레포로 **CLAUDE.md 작성** | 내 프로젝트 룰 파일 |
| Lab 2 | **내 하네스 만들기** — 스킬 + 훅 + 플러그인 | `harness/` 플러그인 |
| Lab 3 | 같은 레포에서 **심어둔 결함 찾기** | 회수율 측정표 |
| Lab 4 | **미니 하네스 직접** (파이썬 · NVIDIA Qwen) | 하네스 내부를 손으로 |
| Lab 5 | **내 하네스로 앱 뼈대** (Next.js · Neon · Qwen) | 대화되는 웹앱 |

**두 개가 이틀에 걸쳐 자란다**

```
harness/          ← 오늘 만든다.  내일 팀(agents/)을 얹는다
mcp-host/         ← 오늘 뼈대.    내일 도구·루프·승인을 붙인다
```

**사전 준비** — Claude Code 로그인 (Pro/Max/Team/Console) · NVIDIA API 키 · [neon.tech](https://neon.tech) 계정.
설치는 `day08/Claude_Code_Codex_세팅.md` 참고.


## Lab 1 · buggy-request 레포로 CLAUDE.md 작성

완벽한 룰 파일이 목표가 아니다. **초안을 빠르게 뽑고, 실수마다 한 줄씩 추가**하는 게 정석이다.

### 1. 클론 → /init

```bash
git clone https://github.com/TunaLee/buggy-request.git
cd buggy-request
claude
```

```
> /init
  코드베이스 분석 → CLAUDE.md 초안 자동 생성
```

### 2. /memory 로 로드 확인

```
> /memory
  로드된 파일 목록 · 자동 메모리 ON/OFF
```

목록에 CLAUDE.md가 **없으면** 위치·파일명 문제다 — Claude가 아예 못 본 것.

### 3. 5섹션으로 정리

```markdown
# 스택
Next.js 15 (App Router) · TypeScript · Prisma + Postgres

# 디렉토리
src/app/   페이지·API
prisma/    스키마·migration
tests/     vitest 단위·통합

# 명령어
npm run dev · npm test · npx prisma migrate dev

# 룰
- server component 우선 (use client 최소화)
- import 절대경로 @/...
- 새 API는 zod로 입력 검증

# 하지 말 것
- .env / .env.local 커밋
- any 사용
- 테스트 없이 라우트 추가
```

### 4. 같은 태스크를 전/후로 비교

`"테스트 1개 추가해줘"` 를 **CLAUDE.md 없이 / 있이** 각각 시켜본다. 무엇이 달라지는가?

### 5. 빗나간 부분 → 룰 한 줄로 환원

- `"any 썼네"` → **하지 말 것**에 추가
- `"테스트 위치 틀렸네"` → **디렉토리**에 추가

### 막혔을 때

한 줄도 안 떠오르면 — **최근 에이전트가 가장 짜증나게 했던 실수 1개**만 떠올려 "하지 말 것"에 적는다. 거기서부터 거꾸로 채워진다.

### 원리

CLAUDE.md는 **강제가 아니라 컨텍스트**다. 따르려고 읽는 것일 뿐, 반드시 막아야 하면 **Hook**으로 (9강).

마지막에 `git commit` — 다음 세션부터 **프롬프트 캐싱**이 걸린다.


## Lab 2 · 내 하네스 만들기 — 스킬 · 훅 · 플러그인

CH5에서 배운 확장 세 가지를 **직접 만든다.** 내일 여기에 팀을 얹는다.

```
harness/
├── .claude-plugin/plugin.json
├── skills/our-rules/SKILL.md      ← 규약 (부탁)
└── hooks/{hooks.json, guard.mjs}  ← 강제 (차단)
```

### 1. 스킬 — 규약을 파일로

`harness/skills/our-rules/SKILL.md`:

```markdown
---
name: our-rules
description: 이 프로젝트의 규약. 코드를 쓰기 전에 읽는다.
---

- API 키는 절대 클라이언트 코드에 두지 않는다 (서버 전용)
- DB 접근은 Prisma로만 — 생 SQL 금지
- 새 API Route는 입력 검증을 붙인다
- 만든 코드는 반드시 실행해서 확인한다
```

> **스킬의 정체는 프론트매터 + 마크다운이 전부다.** 마법이 아니다.

### 2. 훅 — 강제를 코드로

`harness/hooks/guard.mjs` (ESM — `import`, `require` 금지):

```javascript
import { readFileSync } from 'node:fs';
const input = JSON.parse(readFileSync(0, 'utf8'));
const p = (input.tool_input?.file_path || '').replace(/\\/g, '/').toLowerCase();
const cmd = input.tool_input?.command || '';
const BLOCK = ['.env', '/.git/', 'id_rsa', 'secrets', '.pem'];

if (BLOCK.some(b => p.includes(b)) || /rm\s+-rf\s+[\/~]/.test(cmd)) {
  console.error('차단: 민감 경로 / 위험 명령');
  process.exit(2);          // 2 = 도구 호출 차단
}
process.exit(0);
```

`harness/hooks/hooks.json`:

```json
{ "hooks": { "PreToolUse": [{
  "matcher": "Read|Edit|Write|Bash",
  "hooks": [{ "type": "command",
              "command": "node ${CLAUDE_PLUGIN_ROOT}/hooks/guard.mjs" }]
}]}}
```

**단독 테스트 — 진짜 막히는지 먼저 확인한다:**

```bash
echo '{"tool_input":{"file_path":"x/.env"}}' | node harness/hooks/guard.mjs; echo $?
# → 2   (2가 안 나오면 훅이 동작 안 한다)
```

### 3. 플러그인으로 묶기

`harness/.claude-plugin/plugin.json`:

```json
{ "name": "harness", "version": "0.1.0",
  "description": "내 프로젝트 하네스 — 규약 스킬 + 안전 훅" }
```

```bash
claude plugin validate ./harness
claude --plugin-dir ./harness
```

### 4. 확인 — 오늘 만든 게 오늘 일한다

`claude --plugin-dir ./harness` 로 켜고 시켜본다:

```
.env 파일을 읽어서 내용 보여줘
```

**막히면 성공이다.** 훅이 `exit 2`로 도구 호출을 차단한다.

### 원리

| 수단 | 성격 | 무시될 수 있나 |
|---|---|---|
| **CLAUDE.md · 스킬** | 컨텍스트 — 따르려고 읽는다 | **그렇다** |
| **훅** | 강제 — 도구 호출 직전 차단 | 아니다 |

> **스킬은 부탁, 훅은 강제.** Replit 사고를 막는 건 후자다.


## Lab 3 · 심어둔 결함 찾기 (회수율 측정)

Lab 1에서 클론한 **그 buggy-request 레포 그대로**. 곳곳에 결함이 숨어 있다 — 보안 · 로직 · 런타임.

작성한 CLAUDE.md를 컨텍스트로, **어떤 도구 조합이 결함을 얼마나 잡아내는지 직접 측정**한다.

### 측정 절차 — 변수를 하나씩만 바꾼다

| 단계 | 명령 | 찾은 결함 수 |
|---|---|---|
| ① | `/review` 단독 | ___ 개 |
| ② | `+ /cso` (보안 특화) | ___ 개 |
| ③ | `+ 테스트 · /qa` (런타임) | ___ 개 |

```
> /review
  ... 발견 항목 기록

> /cso
  보안 특화 점검 — 회수율 변화 확인

> /qa
  런타임에서만 드러나는 결함 (off-by-one · 상태 전이 · 삭제 미반영)
```

### 왜 이렇게 하나

어떤 도구·조합이 회수율을 끌어올리는지는 **감이 아니라 숫자로** 확인한다. 변수를 하나씩만 바꿔 재측정하면 차이가 드러난다.

### 리스크 티어 → 명령 매핑

| 리스크 | 명령 |
|---|---|
| 낮음 | `/code-review` · `/qa` |
| 중간 | + `/security-review` · `/review` |
| 높음 | + `/cso` · **사람 최종 승인** |

### 원리

에이전트는 **제안이 아니라 실행**한다. 삼성(데이터 유출) · EchoLeak(프롬프트 인젝션) · Replit(폭주) — 셋 다 나중에 막을 문제가 아니라 **설계에서 막을 문제**였다.

> 이 회수율 측정 방식이 곧 **도입 보고서의 근거**가 된다 (9강 도입 로드맵).


## Lab 4 · 미니 하네스 직접 (파이썬 · NVIDIA Qwen)

Claude Code의 **내부**를 손으로 짠다. 하네스는 결국 네 조각이다:

**도구 레지스트리 · 에이전트 루프 · 컨텍스트 트리밍 · 확장점(스킬 로더)**

> Lab 2에서 Claude Code를 **확장**했다면, 여기선 하네스 자체를 **만든다**.
> 이걸 해봐야 내일 메타 하네스가 찍어낸 걸 **읽을 수 있다**.


In [ ]:
%pip install -q openai

In [ ]:
# Lab 4(미니 하네스)에서 LLM으로 Qwen을 부른다. 토큰은 입력창/환경변수 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)
_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]


In [ ]:
# Lab 4 — 미니 하네스 = 도구 레지스트리 + 에이전트 루프 + 컨텍스트 트리밍 + 확장점(스킬)
class MiniHarness:
    """'미니 Claude Code' — 하네스의 네 조각을 직접 조립한다."""
    def __init__(self, system, keep=10):
        self.system = system
        self.registry = {}      # 이름 → 함수
        self.specs = []         # OpenAI 도구 스펙
        self.skills = {}        # 확장점: 트리거 단어 → 노하우
        self.keep = keep        # 컨텍스트로 남길 최근 메시지 수

    def tool(self, description, params):                 # 도구 등록 데코레이터
        def deco(fn):
            self.registry[fn.__name__] = fn
            self.specs.append({"type": "function", "function": {
                "name": fn.__name__, "description": description,
                "parameters": {"type": "object",
                               "properties": {k: {"type": v} for k, v in params.items()},
                               "required": list(params)}}})
            return fn
        return deco

    def skill(self, trigger, text):                      # 확장점: 트리거 단어가 나오면 노하우 주입
        self.skills[trigger] = text

    def _system(self, q):
        extra = [t for k, t in self.skills.items() if k in q]
        return self.system + ("\n\n[스킬]\n" + "\n".join(extra) if extra else "")

    def _trim(self, messages):                           # 컨텍스트 트리밍: system + 최근 keep개
        if len(messages) <= self.keep + 1:
            return messages
        return messages[:1] + messages[-self.keep:]

    def run(self, question, max_steps=6):
        messages = [{"role": "system", "content": self._system(question)},
                    {"role": "user", "content": question}]
        for _ in range(max_steps):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=self._trim(messages), tools=self.specs,
                temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"  [도구] {tc.function.name}({args})")
                try:
                    out = self.registry[tc.function.name](**args)
                except Exception as e:
                    out = f"오류: {e}"
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(out, ensure_ascii=False, default=str)})


In [ ]:
# 미니 하네스에 도구 2개를 등록하고, 스킬(확장점)을 하나 붙여 돌려본다
harness = MiniHarness("너는 도구를 쓰는 조수다. 한국어로 간결히 답하라.")

@harness.tool(description="두 정수를 더한다", params={"a": "integer", "b": "integer"})
def add(a, b):
    return a + b

@harness.tool(description="도시의 (예시) 날씨를 반환한다", params={"city": "string"})
def weather(city):
    return {"city": city, "temp": 21, "sky": "맑음"}

harness.skill("날씨", "날씨를 물으면 weather 도구를 쓰고 섭씨로 답한다.")   # 확장점(스킬) 주입



## Lab 5 · 내 하네스로 앱 뼈대 — MCP 호스트

**보일러플레이트 없이 맨바닥부터.** 오늘 끝나면 브라우저에서 **Qwen과 대화**된다.

> 직접 타이핑하지 않는다. **Claude Code에게 시키고, 내가 판정한다.**

---

### 사전 준비 두 가지

**1. Neon DB** — [neon.tech](https://neon.tech) 가입 → 프로젝트 생성 → **Connection string** 복사
```
postgresql://user:pw@ep-xxx.ap-southeast-1.aws.neon.tech/neondb?sslmode=require
```

**2. NVIDIA API 키** — 7강에서 쓰던 `nvapi-...` 그대로

---

### Step 1 · 프로젝트 생성

```bash
npx create-next-app@latest mcp-host --typescript --tailwind --app --no-src-dir
cd mcp-host
npm install openai prisma @prisma/client
npx prisma init --datasource-provider postgresql
claude
```

Claude Code를 켜고 **먼저 `/init`** 으로 CLAUDE.md를 만든다 (Lab 1에서 한 그대로).

### Step 2 · 환경변수 — 키는 서버에만

`.env` (git에 **절대** 안 올린다):

```bash
DATABASE_URL="postgresql://...neon.tech/neondb?sslmode=require"
NVIDIA_API_KEY="nvapi-xxxxxxxx"
```

`.gitignore` 에 `.env` 가 있는지 **반드시 확인**한다.

> `NEXT_PUBLIC_` 접두사가 붙은 것만 브라우저에 노출된다. **키에는 절대 붙이지 않는다.**

### Step 3 · Prisma 스키마 → Neon에 실제 테이블

Claude Code에 시킨다:

```
prisma/schema.prisma 에 대화 저장용 모델을 만들어줘.

- Conversation: id(cuid), title(기본 "새 대화"), createdAt, messages 관계
- Message: id(cuid), role(user|assistant|tool), content, createdAt,
           conversationId → Conversation 관계 (onDelete: Cascade)

그다음 npx prisma migrate dev --name init 을 실행해서
Neon에 실제 테이블을 만들어줘.
```

**확인** — Neon 콘솔의 Tables 탭에 `Conversation`·`Message` 가 보이면 성공이다.

### Step 4 · API Route — NVIDIA Qwen 스트리밍

```
app/api/chat/route.ts 를 만들어줘.

- POST로 { messages } 를 받는다
- NVIDIA API (baseURL: https://integrate.api.nvidia.com/v1) 로 Qwen 을 부른다
  model: qwen/qwen3-next-80b-a3b-instruct
- 키는 process.env.NVIDIA_API_KEY — 절대 클라이언트로 보내지 마
- stream: true 로 받아서 ReadableStream 으로 토큰을 흘려보낸다
```

나와야 할 뼈대:

```typescript
import OpenAI from "openai";

const llm = new OpenAI({
  baseURL: "https://integrate.api.nvidia.com/v1",
  apiKey: process.env.NVIDIA_API_KEY,      // 서버 전용
});

export async function POST(req: Request) {
  const { messages } = await req.json();

  const stream = await llm.chat.completions.create({
    model: "qwen/qwen3-next-80b-a3b-instruct",
    messages,
    stream: true,
    temperature: 0.2,
  });

  const encoder = new TextEncoder();
  return new Response(
    new ReadableStream({
      async start(controller) {
        for await (const chunk of stream) {
          const delta = chunk.choices[0]?.delta?.content ?? "";
          if (delta) controller.enqueue(encoder.encode(delta));
        }
        controller.close();
      },
    }),
    { headers: { "Content-Type": "text/plain; charset=utf-8" } }
  );
}
```

### Step 5 · 채팅 UI — 스트리밍을 받는다

```
app/page.tsx 를 채팅 UI로 만들어줘.

- "use client" — 입력을 받아야 하니 Client Component
- 메시지 목록 + 입력창 + 전송 버튼
- /api/chat 으로 POST 하고, 스트림을 reader로 읽어
  글자가 타이핑되듯 붙게 해줘
- Tailwind로 간단히
```

### Step 6 · 확인

```bash
npm run dev     # http://localhost:3000
```

- [ ] 질문을 넣으면 **글자가 흘러나온다** (한 번에 안 나온다)
- [ ] 개발자도구 Network 탭에서 **API 키가 안 보인다**
- [ ] Neon 콘솔에 테이블이 있다

---

### 판정 — 내가 짠 것과 비교한다

| 항목 | 확인 |
|---|---|
| 키가 서버에만 있나 | `app/page.tsx` 에 `nvapi-` 가 있으면 **실패** |
| 스트리밍이 되나 | 10초 멈췄다 한 번에 나오면 실패 |
| Prisma 스키마가 맞나 | 관계·onDelete 가 빠지지 않았나 |
| `.env` 가 커밋됐나 | `git status` 에 뜨면 **즉시 수정** |

### 원리

**브라우저는 MCP 서버를 못 붙인다.** 그래서 백엔드가 필요하고, 백엔드가 있으니 **키를 숨길 수 있고**, 그래서 **에이전트 루프도 거기서 돈다.**

> 오늘은 **LLM과 대화만** 된다. 내일 여기에 **도구·루프·승인 게이트**를 붙여 진짜 MCP 호스트로 만든다.


## 산출물 체크리스트

**하네스를 다룬다**
- [ ] `/init` 으로 CLAUDE.md 초안을 뽑고 5섹션으로 정리했다 (Lab 1)
- [ ] 같은 태스크를 **CLAUDE.md 전/후로 비교**했다
- [ ] `/review` → `+/cso` → `+/qa` 로 **회수율을 3번 측정**했다 (Lab 3)

**하네스를 만든다**
- [ ] 스킬(SKILL.md) · 훅(guard.mjs) · 플러그인(harness/)을 **직접 만들었다** (Lab 2)
- [ ] `echo … | node guard.mjs; echo $?` 가 **2**를 뱉는 걸 확인했다
- [ ] `.env` 를 읽게 시켰더니 **훅이 막았다**
- [ ] 파이썬으로 **미니 하네스 내부**를 짰다 — 레지스트리·루프·트리밍·스킬로더 (Lab 4)

**그 하네스로 짓는다** (Lab 5)
- [ ] `claude --plugin-dir ./harness` 로 **내 하네스를 켜고** 시작했다
- [ ] Next.js를 **스크래치부터** 만들고 **Neon**에 `prisma migrate` 했다
- [ ] NVIDIA Qwen을 **서버에서만** 부른다 (프론트에 키가 없다)
- [ ] 응답이 **스트리밍**된다
- [ ] 에이전트가 `.env` 를 건드리려 할 때 **내 훅이 막았다**

> 한 줄: **하네스를 짓고, 그 하네스로 지었다. 내일은 팀을 얹는다.**
